The goal of this project is to create a blackjack solver. For each hand versus the dealer's upcard, the solver will determine the optimal action.
First of all lets start with implementing the basic hit or stand. Aces are going to count as 1 always for now. We will implement the other actions later.

In [7]:
import random
from collections import defaultdict

CARDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 10, 10, 10]
ACTIONS = ("hit", "stand")


def draw_card():
    return random.choice(CARDS)


def dealer_total(dealer_upcard):
    total = dealer_upcard + draw_card()
    while total < 17:
        total += draw_card()
    return total


def reward_for_standing(player_total, dealer_final_total):
    if dealer_final_total > 21 or player_total > dealer_final_total:
        return 1.0
    if player_total < dealer_final_total:
        return -1.0
    return 0.0


def choose_training_action(q_table, state, epsilon):
    if random.random() < epsilon:
        return random.choice(ACTIONS)
    return max(ACTIONS, key=lambda action: q_table[state][action])


def train_q_learning(
    episodes=100_000,
    learning_rate=0.1,
    discount_factor=1.0,
    start_epsilon=1.0,
    end_epsilon=0.05,
):
    q_table = defaultdict(lambda: {action: 0.0 for action in ACTIONS})

    for episode in range(episodes):
        player_total = draw_card() + draw_card()
        dealer_upcard = draw_card()
        epsilon = max(
            end_epsilon,
            start_epsilon - (start_epsilon - end_epsilon) * episode / episodes,
        )

        while True:
            state = (player_total, dealer_upcard)
            action = choose_training_action(q_table, state, epsilon)

            if action == "hit":
                player_total += draw_card()
                if player_total > 21:
                    reward = -1.0
                    next_value = 0.0
                    done = True
                else:
                    reward = 0.0
                    next_state = (player_total, dealer_upcard)
                    next_value = max(q_table[next_state].values())
                    done = False
            else:
                reward = reward_for_standing(
                    player_total,
                    dealer_total(dealer_upcard),
                )
                next_value = 0.0
                done = True

            old_value = q_table[state][action]
            target = reward + discount_factor * next_value
            q_table[state][action] = old_value + learning_rate * (target - old_value)

            if done:
                break

    return q_table


def choose_action(q_table, player_total, dealer_upcard):
    state = (player_total, dealer_upcard)
    values = q_table[state]
    action = max(ACTIONS, key=lambda candidate: values[candidate])
    return action, values


random.seed(42)
q_table = train_q_learning()

for (player_total, dealer_upcard) in sorted(q_table.keys()):
    action, values = choose_action(q_table, player_total, dealer_upcard)
    print(
        f"Player total {player_total} vs dealer {dealer_upcard}: "
        f"{action} | Q-values: {values}"
    )

Player total 2 vs dealer 1: hit | Q-values: {'hit': -0.11596119913156207, 'stand': -0.22279182888735105}
Player total 2 vs dealer 2: hit | Q-values: {'hit': -0.04885963866948223, 'stand': -0.112808344301}
Player total 2 vs dealer 3: hit | Q-values: {'hit': -0.06332856785082336, 'stand': -0.14041710611}
Player total 2 vs dealer 4: hit | Q-values: {'hit': -0.014251854184379627, 'stand': -0.06559186410999998}
Player total 2 vs dealer 5: hit | Q-values: {'hit': -0.050146531733104266, 'stand': -0.2549594048329}
Player total 2 vs dealer 6: hit | Q-values: {'hit': 0.061113153147759405, 'stand': -0.24046702020463237}
Player total 2 vs dealer 7: hit | Q-values: {'hit': -0.028784059534224738, 'stand': -0.2026999751671}
Player total 2 vs dealer 8: hit | Q-values: {'hit': -0.04140897905190257, 'stand': -0.467834775499}
Player total 2 vs dealer 9: stand | Q-values: {'hit': -0.1714064814952803, 'stand': -0.13548530611000004}
Player total 2 vs dealer 10: hit | Q-values: {'hit': -0.31173278780507224, 